# Feature Engineering for Image Segmentation

This notebook applies the handcrafted feature maps from **Chapter 7** to real urothelial cell images from the Cedar database. For each filter, we show how a fixed kernel transforms a raw image into a feature map that emphasizes a specific signal — the same operation a CNN layer performs, but with learned weights instead of hand-designed ones.

**Filters covered:**
1. Raw intensity and color channels
2. Gaussian blur and mean blur
3. Median filter
4. Sobel gradient magnitude
5. Gabor filter bank (4 orientations)
6. GLCM texture features (sliding window)

**Data:** 5 randomly selected images from the Cedar dataset (`imagedata/X/*.npy`).
Each image is shape `(3, 256, 256)` float32, values in [0, 1].
Ground-truth masks (`imagedata/y/*.npy`) label each pixel: 0=background, 1=cytoplasm, 2=nucleus.

In [ ]:
!pip install scikit-image --quiet

# Clone the repo
!git clone https://github.com/emilsar/VocEd.git
%cd VocEd

## Setup

Run this notebook in Google Colab using the badge above, or locally from the **VocEd** repository root.

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.ndimage import uniform_filter, median_filter, sobel, gaussian_filter
from skimage.filters import gabor
from skimage.feature import graycomatrix, graycoprops

# Load all images and masks
N = len(glob.glob('imagedata/X/*.npy'))
print(f'Found {N} images')

X = np.stack([np.load(f'imagedata/X/{i}.npy') for i in range(N)])  # (N, 3, 256, 256) float32
y = np.stack([np.load(f'imagedata/y/{i}.npy') for i in range(N)])  # (N, 256, 256) int64
print(f'X shape: {X.shape}, dtype: {X.dtype}, range: [{X.min():.3f}, {X.max():.3f}]')
print(f'y shape: {y.shape}, labels: {np.unique(y)} (0=bg, 1=cyto, 2=nucleus)')

In [ ]:
# Pick 5 images that each contain all three tissue classes
np.random.seed(7)
has_all = np.array([
    (y[i] == 0).any() and (y[i] == 1).any() and (y[i] == 2).any()
    for i in range(N)
])
candidates = np.where(has_all)[0]
IDX = np.random.choice(candidates, size=5, replace=False)
print('Selected image indices:', IDX)

imgs  = X[IDX]   # (5, 3, 256, 256)
masks = y[IDX]   # (5, 256, 256)

# Grayscale helper (luminance)
def to_gray(img):
    """(3, H, W) float32  ->  (H, W) float32 grayscale."""
    return 0.299*img[0] + 0.587*img[1] + 0.114*img[2]

grays = np.stack([to_gray(img) for img in imgs])   # (5, 256, 256)

---
## Helper: display grid

We'll reuse one plotting function throughout: rows = images, columns = variants.

In [ ]:
def show_grid(row_data, row_titles, col_titles, cmap='gray', vmin=None, vmax=None, figsize=None):
    """
    row_data   : list of lists — row_data[r][c] is the 2D array to display at row r, col c
    row_titles : list of strings, one per row
    col_titles : list of strings, one per column
    """
    nrows = len(row_data)
    ncols = len(row_data[0])
    if figsize is None:
        figsize = (3 * ncols, 3 * nrows)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    if nrows == 1:
        axes = axes[np.newaxis, :]
    if ncols == 1:
        axes = axes[:, np.newaxis]
    for r in range(nrows):
        for c in range(ncols):
            ax = axes[r, c]
            data = row_data[r][c]
            if data.ndim == 3:          # RGB
                ax.imshow(data.transpose(1, 2, 0))
            else:
                ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
            ax.axis('off')
            if r == 0:
                ax.set_title(col_titles[c], fontsize=10, fontweight='bold')
        axes[r, 0].set_ylabel(row_titles[r], fontsize=9, rotation=90, labelpad=4)
    plt.tight_layout()
    plt.show()

---
## 1. Raw Intensity and Color Channels

Before any filtering, inspect what each channel provides. The RGB image, the grayscale luminance projection, and the ground-truth mask are shown side by side.

**Key question:** does a single channel already separate nucleus, cytoplasm, and background?

In [ ]:
MASK_COLORS = plt.cm.get_cmap('viridis', 3)   # 0=bg, 1=cyto, 2=nucleus

row_data = [
    [imgs[i], grays[i], imgs[i][0], imgs[i][1], imgs[i][2], masks[i]]
    for i in range(5)
]
col_titles = ['RGB', 'Grayscale', 'R channel', 'G channel', 'B channel', 'GT mask']
row_titles = [f'Image {IDX[i]}' for i in range(5)]

fig, axes = plt.subplots(5, 6, figsize=(18, 15))
for r in range(5):
    for c, (data, title) in enumerate(zip(row_data[r], col_titles)):
        ax = axes[r, c]
        if c == 5:   # mask
            ax.imshow(data, cmap='viridis', vmin=0, vmax=2, interpolation='nearest')
        elif c == 0: # RGB
            ax.imshow(data.transpose(1,2,0))
        else:
            ax.imshow(data, cmap='gray', vmin=0, vmax=1)
        ax.axis('off')
        if r == 0:
            ax.set_title(title, fontsize=11, fontweight='bold')
    axes[r, 0].set_ylabel(f'Image {IDX[r]}', fontsize=9)
plt.suptitle('Raw channels  |  mask: 0=background (purple), 1=cytoplasm (teal), 2=nucleus (yellow)', fontsize=12)
plt.tight_layout()
plt.show()

---
## 2. Gaussian Blur and Mean Blur

Both filters replace each pixel with a weighted average of its neighborhood:

$$F(x,y) = (K * I)(x,y) = \sum_{u,v} K(u,v)\, I(x-u, y-v)$$

**Mean blur** weights all neighbors equally. **Gaussian blur** weights the center more than the periphery.  
Both reduce noise; Gaussian blurs more gracefully (no ringing artifacts).

In [ ]:
mean3   = np.stack([uniform_filter(g, size=3)  for g in grays])
mean9   = np.stack([uniform_filter(g, size=9)  for g in grays])
gauss1  = np.stack([gaussian_filter(g, sigma=1) for g in grays])
gauss3  = np.stack([gaussian_filter(g, sigma=3) for g in grays])

fig, axes = plt.subplots(5, 5, figsize=(16, 12))
col_titles = ['Grayscale', 'Mean 3×3', 'Mean 9×9', 'Gaussian σ=1', 'Gaussian σ=3']
data_cols  = [grays, mean3, mean9, gauss1, gauss3]
for r in range(5):
    for c in range(5):
        ax = axes[r, c]
        ax.imshow(data_cols[c][r], cmap='gray', vmin=0, vmax=1)
        ax.axis('off')
        if r == 0:
            ax.set_title(col_titles[c], fontsize=10, fontweight='bold')
    axes[r, 0].set_ylabel(f'Image {IDX[r]}', fontsize=9)
plt.suptitle('Gaussian and Mean Blur', fontsize=13)
plt.tight_layout()
plt.show()

---
## 3. Median Filter

The median filter replaces each pixel with the **median** of its neighborhood — it cannot be written as a convolution. It is highly effective at removing impulse (salt-and-pepper) noise while preserving edges better than a mean filter.

In [ ]:
med3 = np.stack([median_filter(g, size=3) for g in grays])
med7 = np.stack([median_filter(g, size=7) for g in grays])

# Add synthetic impulse noise for comparison
rng = np.random.default_rng(42)
grays_noisy = grays.copy()
noise_mask = rng.random(grays.shape) < 0.03
grays_noisy[noise_mask] = rng.choice([0.0, 1.0], size=noise_mask.sum())
med3_noisy = np.stack([median_filter(g, size=3) for g in grays_noisy])
mean3_noisy = np.stack([uniform_filter(g, size=3) for g in grays_noisy])

fig, axes = plt.subplots(5, 5, figsize=(16, 12))
col_titles = ['Original', 'Median 3×3', 'Median 7×7', 'Noisy input', 'Median 3×3\n(on noisy)']
data_cols  = [grays, med3, med7, grays_noisy, med3_noisy]
for r in range(5):
    for c in range(5):
        ax = axes[r, c]
        ax.imshow(data_cols[c][r], cmap='gray', vmin=0, vmax=1)
        ax.axis('off')
        if r == 0:
            ax.set_title(col_titles[c], fontsize=10, fontweight='bold')
    axes[r, 0].set_ylabel(f'Image {IDX[r]}', fontsize=9)
plt.suptitle('Median Filter — edge-preserving noise removal', fontsize=13)
plt.tight_layout()
plt.show()

---
## 4. Sobel Gradient Magnitude

Two fixed derivative kernels detect intensity change along each axis:

$$K_x = \begin{bmatrix}-1&0&1\\-2&0&2\\-1&0&1\end{bmatrix}, \quad
K_y = \begin{bmatrix}-1&-2&-1\\0&0&0\\1&2&1\end{bmatrix}$$

$$G_x = K_x * I, \quad G_y = K_y * I, \quad M = \sqrt{G_x^2 + G_y^2}$$

Bright pixels in $M$ mark cell boundaries and nucleus edges.

In [ ]:
def sobel_magnitude(gray):
    gx = sobel(gray, axis=1)   # horizontal derivative
    gy = sobel(gray, axis=0)   # vertical derivative
    mag = np.hypot(gx, gy)
    return mag / mag.max() if mag.max() > 0 else mag

# Apply on both raw and pre-smoothed grayscale
sobel_raw    = np.stack([sobel_magnitude(g) for g in grays])
sobel_smooth = np.stack([sobel_magnitude(gaussian_filter(g, sigma=1)) for g in grays])

fig, axes = plt.subplots(5, 4, figsize=(13, 12))
col_titles = ['Grayscale', 'Sobel (raw)', 'Gaussian σ=1', 'Sobel (after Gaussian)']
data_cols  = [grays, sobel_raw, gauss1, sobel_smooth]
for r in range(5):
    for c in range(5):
        if c >= 4:
            break
        ax = axes[r, c]
        ax.imshow(data_cols[c][r], cmap='gray' if c != 1 and c != 3 else 'hot', vmin=0, vmax=1)
        ax.axis('off')
        if r == 0:
            ax.set_title(col_titles[c], fontsize=10, fontweight='bold')
    axes[r, 0].set_ylabel(f'Image {IDX[r]}', fontsize=9)
plt.suptitle('Sobel Gradient Magnitude  (hot colormap = stronger edge response)', fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. Gabor Filter Bank

A Gabor filter is a Gaussian-modulated sinusoid:

$$G_{\theta,\lambda,\sigma}(x,y) = \exp\!\left(-\frac{x'^2 + \gamma^2 y'^2}{2\sigma^2}\right) \cos\!\left(\frac{2\pi x'}{\lambda}\right)$$

where $x' = x\cos\theta + y\sin\theta$.  It fires when it encounters **periodic texture** at orientation $\theta$ and spatial frequency $1/\lambda$.

We apply a **filter bank** at 4 orientations: 0°, 45°, 90°, 135°. Each produces one feature map. Together they capture directional texture differences between nucleus, cytoplasm, and background.

In [ ]:
FREQ   = 0.15          # spatial frequency (cycles/pixel)
THETAS = [0, np.pi/4, np.pi/2, 3*np.pi/4]   # 0°, 45°, 90°, 135°
THETA_LABELS = ['0°', '45°', '90°', '135°']

def gabor_magnitude(gray, theta, frequency=FREQ):
    real, imag = gabor(gray, frequency=frequency, theta=theta)
    mag = np.hypot(real, imag)
    return mag / mag.max() if mag.max() > 0 else mag

# Shape: (5 images, 4 orientations, 256, 256)
gabor_maps = np.stack([
    np.stack([gabor_magnitude(g, th) for th in THETAS])
    for g in grays
])

fig, axes = plt.subplots(5, 5, figsize=(17, 13))
col_titles = ['Grayscale'] + [f'Gabor {lbl}' for lbl in THETA_LABELS]
for r in range(5):
    axes[r, 0].imshow(grays[r], cmap='gray', vmin=0, vmax=1)
    axes[r, 0].axis('off')
    for c in range(4):
        axes[r, c+1].imshow(gabor_maps[r, c], cmap='magma', vmin=0, vmax=1)
        axes[r, c+1].axis('off')
    axes[r, 0].set_ylabel(f'Image {IDX[r]}', fontsize=9)
for c, title in enumerate(col_titles):
    axes[0, c].set_title(title, fontsize=10, fontweight='bold')
plt.suptitle(f'Gabor Filter Bank  (frequency={FREQ}, 4 orientations)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Overlay: show nucleus/cytoplasm/background mean Gabor response per orientation
print('Mean Gabor response by tissue class and orientation:')
print(f'{"":12}  {"  ".join([f"Gabor {lbl}":>10s} for lbl in THETA_LABELS]}')
class_names = ['Background', 'Cytoplasm ', 'Nucleus   ']
for cls in range(3):
    means = []
    for t in range(4):
        vals = [gabor_maps[i, t][masks[i] == cls].mean() for i in range(5)]
        means.append(np.mean(vals))
    row = '  '.join([f'{m:10.4f}' for m in means])
    print(f'{class_names[cls]:12}  {row}')

---
## 6. GLCM Texture Features (Sliding Window)

The Gray Level Co-Occurrence Matrix counts how often intensity pair $(i, j)$ appears at offset $\Delta$:

$$C_{\Delta}(i,j) = \#\{(r,c) : I(r,c)=i \text{ and } I(r+\Delta r, c+\Delta c)=j\}$$

From the normalized GLCM we extract **Haralick features**:

| Feature | Captures |
|---|---|
| Contrast | Local intensity variation |
| Energy | Textural uniformity |
| Homogeneity | Smoothness of the GLCM diagonal |
| Correlation | Linear gray-level dependency |

For segmentation we compute these **in a sliding window** — each pixel gets a local feature value.

In [ ]:
from skimage.util import view_as_windows

def glcm_feature_map(gray, win=15, levels=32, feature='contrast'):
    """
    Compute a scalar Haralick feature map by sliding a window over the image.
    gray   : (H, W) float32  in [0, 1]
    win    : side length of the sliding window (must be odd)
    levels : number of gray levels to quantize to
    feature: 'contrast' | 'energy' | 'homogeneity' | 'correlation'
    Returns (H, W) float32 feature map.
    """
    H, W = gray.shape
    pad  = win // 2
    # quantize to [0, levels-1]
    img_q = np.clip((gray * (levels - 1)).astype(np.uint8), 0, levels - 1)
    # pad with reflection
    img_p = np.pad(img_q, pad, mode='reflect')
    out   = np.zeros((H, W), dtype=np.float32)
    for r in range(H):
        for c in range(W):
            patch = img_p[r:r+win, c:c+win]
            glcm  = graycomatrix(patch, distances=[1], angles=[0],
                                 levels=levels, symmetric=True, normed=True)
            out[r, c] = graycoprops(glcm, feature)[0, 0]
    return out

# This is slow for full 256×256 — compute on a downsampled version for speed
# then upsample back for display
from skimage.transform import resize

SCALE = 0.25   # compute on 64×64, display at 256×256

def glcm_fast(gray, feature='contrast', win=11, levels=32):
    small = resize(gray, (int(gray.shape[0]*SCALE), int(gray.shape[1]*SCALE)),
                   anti_aliasing=True)
    fmap  = glcm_feature_map(small, win=win, levels=levels, feature=feature)
    fmap  = resize(fmap, gray.shape, anti_aliasing=True)
    # normalize to [0,1] for display
    lo, hi = fmap.min(), fmap.max()
    return (fmap - lo) / (hi - lo + 1e-8)

print('Computing GLCM feature maps (this takes ~30–60 seconds)...')
glcm_contrast     = np.stack([glcm_fast(g, 'contrast')     for g in grays])
glcm_energy       = np.stack([glcm_fast(g, 'energy')       for g in grays])
glcm_homogeneity  = np.stack([glcm_fast(g, 'homogeneity')  for g in grays])
print('Done.')

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(17, 13))
col_titles = ['Grayscale', 'GT mask', 'GLCM Contrast', 'GLCM Energy', 'GLCM Homogeneity']
for r in range(5):
    axes[r, 0].imshow(grays[r],            cmap='gray',   vmin=0, vmax=1)
    axes[r, 1].imshow(masks[r],            cmap='viridis', vmin=0, vmax=2, interpolation='nearest')
    axes[r, 2].imshow(glcm_contrast[r],    cmap='inferno', vmin=0, vmax=1)
    axes[r, 3].imshow(glcm_energy[r],      cmap='inferno', vmin=0, vmax=1)
    axes[r, 4].imshow(glcm_homogeneity[r], cmap='inferno', vmin=0, vmax=1)
    for c in range(5):
        axes[r, c].axis('off')
    axes[r, 0].set_ylabel(f'Image {IDX[r]}', fontsize=9)
for c, t in enumerate(col_titles):
    axes[0, c].set_title(t, fontsize=10, fontweight='bold')
plt.suptitle('GLCM Texture Feature Maps  (computed in 11×11 sliding window on 64×64 then upsampled)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Quantitative: mean GLCM feature value per tissue class
print('Mean GLCM feature value by tissue class (averaged over 5 images):\n')
print(f'{"":12}  {"Contrast":>10}  {"Energy":>10}  {"Homogeneity":>12}')
class_names = ['Background', 'Cytoplasm ', 'Nucleus   ']
for cls in range(3):
    c_vals = [glcm_contrast[i][masks[i]==cls].mean()    for i in range(5)]
    e_vals = [glcm_energy[i][masks[i]==cls].mean()      for i in range(5)]
    h_vals = [glcm_homogeneity[i][masks[i]==cls].mean() for i in range(5)]
    print(f'{class_names[cls]:12}  {np.mean(c_vals):10.4f}  {np.mean(e_vals):10.4f}  {np.mean(h_vals):12.4f}')

---
## Summary: Which Features Best Separate the Three Classes?

The table below collects the mean grayscale intensity and Sobel gradient magnitude for each tissue class, giving a quick sense of which single-channel features are most discriminative.

In [ ]:
print('Mean feature value per tissue class (averaged over 5 images):\n')
print(f'{"":12}  {"Grayscale":>10}  {"Sobel mag":>10}  {"Gabor 0°":>10}  {"Gabor 90°":>10}')
class_names = ['Background', 'Cytoplasm ', 'Nucleus   ']
for cls in range(3):
    g_vals  = [grays[i][masks[i]==cls].mean()         for i in range(5)]
    s_vals  = [sobel_smooth[i][masks[i]==cls].mean()  for i in range(5)]
    g0_vals = [gabor_maps[i,0][masks[i]==cls].mean()  for i in range(5)]
    g2_vals = [gabor_maps[i,2][masks[i]==cls].mean()  for i in range(5)]
    print(
        f'{class_names[cls]:12}  '
        f'{np.mean(g_vals):10.4f}  '
        f'{np.mean(s_vals):10.4f}  '
        f'{np.mean(g0_vals):10.4f}  '
        f'{np.mean(g2_vals):10.4f}'
    )

print('\nObservations:')
print('  - Grayscale alone separates nucleus (dark) from background (bright) well.')
print('  - Cytoplasm sits in between, making pure thresholding ambiguous.')
print('  - Sobel magnitude is highest at class boundaries, not interiors.')
print('  - Gabor features add orientation-specific texture information.')
print('  - GLCM features capture second-order statistics not visible in raw intensity.')
print('  - Combining these feature maps into a multi-channel input to a classifier')
print('    (or CNN) improves segmentation far beyond any single feature alone.')

---
## From Feature Engineering to CNNs

Every filter in this notebook uses a **fixed kernel** — the weights are hand-designed. A CNN replaces those fixed weights with **learned weights**: the network sees the raw image and the ground-truth mask, and through backpropagation it adjusts kernel values until the feature maps are maximally discriminative.

Studies of trained CNNs show that:
- **Early layers** learn filters that closely resemble Gaussian, Sobel, and Gabor kernels.
- **Middle layers** combine these to detect blobs, curves, and junctions.
- **Deep layers** represent high-level structures (cell boundaries, nuclear contours).

This is exactly why classical feature engineering is the right precursor to CNNs — not as an alternative, but as an explanation of what the network is learning to do automatically.